In [27]:
from pathlib import Path
import pandas as pd

# =========================
# 1. CARGA DE PARQUETS
# =========================
path = Path(r"..\..\Raw\EV_prediction_raw_data")
files = list(path.glob("*.parquet"))

dfs = []

for f in files:
    df_temp = pd.read_parquet(f)
    
    # fecha del dataset (opcional, para control)
    year, month = f.stem.split("_")
    df_temp["fecha_dataset"] = pd.to_datetime(f"{year}-{month}-01")
    
    dfs.append(df_temp)

df_ev_raw = pd.concat(dfs, ignore_index=True)

print("Dimensión inicial:", df_ev_raw.shape)

Dimensión inicial: (14588498, 8)


In [28]:
# =========================
# 2. NORMALIZACIÓN DE COLUMNAS
# =========================
df_ev_raw.columns = (
    df_ev_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [29]:
# =========================
# 3. LIMPIEZA BÁSICA DE TEXTO
# =========================
for col in df_ev_raw.select_dtypes(include="object").columns:
    df_ev_raw[col] = (
        df_ev_raw[col]
        .astype(str)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

In [30]:
# =========================
# 4. LIMPIEZA DE FECHA DE MATRICULACIÓN
# =========================

# asegurar formato correcto (DDMMYYYY)
df_ev_raw["fec_matricula"] = (
    df_ev_raw["fec_matricula"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

# convertir a datetime
df_ev_raw["fecha_matricula"] = pd.to_datetime(
    df_ev_raw["fec_matricula"],
    format="%d%m%Y",
    errors="coerce"
)

In [31]:
# =========================
# 5. VARIABLES TEMPORALES
# =========================
df_ev_raw["year"] = df_ev_raw["fecha_matricula"].dt.year
df_ev_raw["month"] = df_ev_raw["fecha_matricula"].dt.month
df_ev_raw["year_month"] = df_ev_raw["fecha_matricula"].dt.to_period("M")

In [32]:
# =========================
# 6. FILTRO DE FECHAS VÁLIDAS
# =========================

df_ev_raw = df_ev_raw[
    df_ev_raw["fecha_matricula"].notna() &
    (df_ev_raw["year"] >= 2000) &
    (df_ev_raw["year"] <= 2026)
]

In [33]:
# =========================
# 7. LIMPIEZA DE DUPLICADOS
# =========================
df_ev_raw = df_ev_raw.drop_duplicates()

In [59]:
# =========================
# 8. LIMPIEZA COLUMNA EV
# =========================

col = "categoría_vehículo_eléctrico"

df_ev_raw[col] = (
    df_ev_raw[col]
    .astype(str)
    .str.strip()
    .str.upper()  # normaliza valores tipo hev, Hev, etc.
    .replace({
        "": pd.NA,
        "NA": pd.NA,
        "N/A": pd.NA,
        "<NA>": pd.NA,
        "NAN": pd.NA,
        "NONE": pd.NA,
        "000": pd.NA,
        "0000": pd.NA,  
        "0": pd.NA,
        "FCEV": pd.NA, # Not plug-in vehicle
        "899": pd.NA, # Not plug-in vehicle
        "NOVC": pd.NA # Not plug-in vehicle
    })
)


In [60]:
# =========================
# 9. FILTRAR SOLO VEHÍCULOS ELÉCTRICOS
# =========================

df_ev = df_ev_raw[
    df_ev_raw["categoría_vehículo_eléctrico"].notna()
].copy()

In [61]:
# =========================
# 10. DATASET FINAL LIMPIO
# =========================

print("Dimensión EV:", df_ev.shape)

df_ev.head()

Dimensión EV: (270213, 12)


,fec_matricula,marca_itv,modelo_itv,cod_tipo,cod_propulsion_itv,clave_tramite,categoría_vehículo_eléctrico,fecha_dataset,fecha_matricula,year,month,year_month
2,02012015,LEXUS,LEXUS GS300H,40,0,1,HEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01
129,02012015,TOYOTA,AURIS,40,0,1,HEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01
372,02012015,LEXUS,LEXUS CT200H,40,0,1,HEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01
474,02012015,LEXUS,LEXUS IS300H,40,0,1,HEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01
530,02012015,VOLKSWAGEN,UP!,40,2,1,BEV,2015-01-01,2015-01-02,2015.0,1.0,2015-01


In [62]:
df_ev["categoría_vehículo_eléctrico"].unique()[:20]

array(['HEV', 'BEV', 'REEV', 'PHEV', 'HVE'], dtype=object)

In [63]:
import os

print("CWD:", os.getcwd())
print("Existe carpeta relativa:", os.path.isdir(r"..\..\Data\Processed\ev_cleaned"))
print("Ruta absoluta:", os.path.abspath(r"..\..\Data\Processed\ev_cleaned"))

CWD: c:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\ev_cleaned
Existe carpeta relativa: False
Ruta absoluta: c:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Data\Processed\ev_cleaned


In [64]:
df_ev.to_parquet(r"..\..\Processed\ev_cleaned\ev_clean.parquet", index=False)